# 04 · Retrieval-augmented generation over your documents ⏱️ 20 minutes

The model knows nothing about your documents, and its context window is small. RAG retrieves the few passages that matter,
puts them in the prompt, and asks for an answer with citations. You will see the retrieved passages **before** the answer,
so you can judge whether the answer had a chance.

Corpus: the IFLA Library Reference Model and the DCMI Metadata Terms (both CC BY 4.0), as text extracts.

**What you'll learn**
1. Load and chunk documents, keeping the source of every chunk
2. Embed once, retrieve by question
3. Prompt for grounded answers with citations and a refusal when the documents are silent
4. Respect the context budget, and measure with five questions

## 04.01 Configuration

In [1]:
import os
os.environ.setdefault("WORKSHOP_TIER", "16gb")
from workshop.config import load_settings, context_limit
settings = load_settings(); MODEL = settings.model; EMBED_MODEL = settings.embed_model
LIMIT = context_limit(MODEL)
K = 3 if LIMIT <= 4096 else 4         # fewer chunks on the 4B's 4k window
print("Model:", MODEL, "| context:", LIMIT, "| chunks per question:", K)

Model: nishad/apertus:8b | context: 8192 | chunks per question: 4


## 04.02 Load and chunk

Chunks of about 300 words with a 50-word overlap. Each chunk remembers its file and position; that is what a citation points to.

In [2]:
from pathlib import Path
from workshop.rag import load_document, chunk_text
chunks = []
for path in sorted(Path("data/docs").glob("*.md")):
    text = load_document(str(path))
    chunks += chunk_text(text, source=path.stem, size=300, overlap=50)
by_id = {c.id: c for c in chunks}
print(len(chunks), "chunks from", len(set(c.source for c in chunks)), "documents")
print(chunks[5].id, "→", chunks[5].text[:200], "...")

173 chunks from 2 documents
dcmi-terms#5 → The distinction between titles and alternative titles is application-specific. Has Range http://www.w3.org/2000/01/rdf-schema#Literal Subproperty of Title (http://purl.org/dc/elements/1.1/title) Title ...


## 04.03 Embed once

A few hundred chunks: this takes under a minute on most laptops. The index is kept in memory for the session.

In [3]:
import time
from workshop.search import build_index, search
t = time.time()
index = build_index([c.id for c in chunks], [c.text for c in chunks], model=EMBED_MODEL)
print(f"indexed {len(chunks)} chunks in {time.time()-t:.0f} s")

indexed 173 chunks in 17 s


## 04.04 Look at what was retrieved before you read any answer

If the right passage is not in the top hits, no prompt can fix it. Retrieval quality comes first.

In [4]:
question = "Which DCMI term expresses the date on which a resource was formally issued?"
for cid, text, score in search(question, index, k=K):
    print(f"{score:.3f}  {cid}\n      {text[:160]}...\n")

0.721  dcmi-terms#7
      Definition A point or period of time associated with an event in the lifecycle of the resource. Comment Date may be used to express temporal information at any ...

0.630  dcmi-terms#8
      property Date, of which this is a subproperty. Has Range http://www.w3.org/2000/01/rdf-schema#Literal Subproperty of Date (http://purl.org/dc/elements/1.1/date)...

0.612  dcmi-terms#6
      resource, spatial applicability of the resource, or jurisdiction under which the resource is relevant. Comment Spatial topic and spatial applicability may be a ...

0.601  dcmi-terms#24
      resource. ### FileFormat Class More details → URI http://purl.org/dc/terms/FileFormat Label File Format Definition A digital resource format. Subclass of http:/...



## 04.05 Ask with the documents in the prompt

The prompt numbers the chunks, tells the model to cite them as [n], and to say when the documents do not answer.
`answer()` returns both the text and the chunks it used, so the citation numbers can be checked.

In [5]:
from workshop.rag import answer, build_prompt
from workshop.display import show_markdown
text, used = answer(question, index, by_id, k=K, model=MODEL)
show_markdown(text)
print("\nCited material:")
for n, c in enumerate(used, 1):
    print(f"[{n}] {c.source}#{c.position}: {c.text[:100]}...")


Cited material:
[1] dcmi-terms#7: Definition A point or period of time associated with an event in the lifecycle of the resource. Comm...
[2] dcmi-terms#8: property Date, of which this is a subproperty. Has Range http://www.w3.org/2000/01/rdf-schema#Litera...
[3] dcmi-terms#6: resource, spatial applicability of the resource, or jurisdiction under which the resource is relevan...
[4] dcmi-terms#24: resource. ### FileFormat Class More details → URI http://purl.org/dc/terms/FileFormat Label File For...


## 04.06 When the documents are silent

A grounded system must refuse. Ask something the corpus cannot know.

In [6]:
text, used = answer("Who won the 2026 Nobel Prize in Literature?", index, by_id, k=K, model=MODEL)
show_markdown(text)

## 04.07 The context budget

On the 8 GB tier the window is 4,096 tokens: system text, retrieved chunks, question and answer must all fit. The helper
estimates the size and refuses to build an oversized prompt; Ollama itself returns an HTTP 400 when a request exceeds the
window. Here is what both look like.

In [7]:
from workshop.rag import PromptTooLong
import ollama
many = [by_id[cid] for cid, _, _ in search(question, index, k=40)]
try:
    build_prompt(question, many, max_tokens=LIMIT)
except PromptTooLong as err:
    print("helper refused:", err)
try:
    big = "\n\n".join(c.text for c in many)
    ollama.Client(host=settings.host).chat(model=MODEL, messages=[{"role": "user", "content": big + "\n\nSummarize."}],
                                          options={"num_ctx": LIMIT, "temperature": 0})
    print("Ollama accepted the long prompt (it may have truncated silently; check num_ctx).")
except ollama.ResponseError as err:
    print("Ollama said:", err)

helper refused: About 25842 tokens of prompt for a 8192-token window. Use fewer or shorter chunks (lower k or size).
Ollama said: {"error":{"code":400,"message":"request (21758 tokens) exceeds the available context size (8192 tokens), try increasing it","type":"exceed_context_size_error","n_prompt_tokens":21758,"n_ctx":8192}} (status code: 400)


## 04.08 Five questions, scored by you

Run the prepared questions and compare with the expected answers. The score is a judgement, not a string match; that is deliberate.

In [8]:
import json
questions = json.loads(Path("data/docs/questions.json").read_text(encoding="utf-8"))
for q in questions:
    text, used = answer(q["question"], index, by_id, k=K, model=MODEL)
    print("Q:", q["question"]); print("A:", text.strip()[:400]); print("expected:", q["expected"]); print("-" * 80)

Q: Which four entities make up the IFLA LRM's work, expression, manifestation, item chain, and what does each one stand for?
A: The four entities that make up the IFLA LRM's work, expression, manifestation, item chain are:

1. Work: Represents the intellectual or artistic content, without consideration of its physical or digital embodiment. It is the abstract concept of the creative work.

2. Expression: Represents the realization of a work in a particular form of language or medium. It is the specific realization of the w
expected: work, expression, manifestation, item (the WEMI entities): the intellectual content, its realization, its physical embodiment, and a single exemplar
--------------------------------------------------------------------------------


Q: Which DCMI term expresses the date on which a resource was formally issued?
A: The DCMI term that expresses the date on which a resource was formally issued is "dateAccepted". This term is defined as "Date of acceptance of the resource. Comment Recommended practice is to describe the date, date/time, or period of time as recommended for the property Date, of which this is a subproperty. Examples of resources to which a date of acceptance may be relevant are a thesis (accepte
expected: dcterms:issued
--------------------------------------------------------------------------------


Q: In the IFLA LRM, what is the difference between a manifestation and an item?
A: In the IFLA LRM, a manifestation is a set of items that reflect the characteristics of that manifestation. An item, on the other hand, is a physical entity that exemplifies a manifestation. The difference between a manifestation and an item is that a manifestation is a set of items, while an item is a single physical entity. For example, a book is a manifestation, and each copy of the book is an i
expected: a manifestation is the set of all carriers sharing the same characteristics; an item is a single exemplar
--------------------------------------------------------------------------------


Q: What is the range of dcterms:language?
A: The range of dcterms:language is Includes http://purl.org/dc/terms/LinguisticSystem [1]. This is specified in the document [1] which provides details on the dcterms:language property. The document [1] also mentions that the recommended practice is to use either a non-literal value representing a language from a controlled vocabulary such as ISO 639-2 or ISO 639-3, or a literal value consisting of 
expected: LinguisticSystem
--------------------------------------------------------------------------------


Q: Who won the 2026 Nobel Prize in Literature?
A: The documents do not answer the question. The question asks who won the 2026 Nobel Prize in Literature, but the documents are about bibliographic identities, context of use, and the recognition of distinct bibliographic identities. They do not contain information about the Nobel Prize in Literature or its winners.
expected: not in the documents (the model must say so)
--------------------------------------------------------------------------------


## 04.09 Bring your own document

Drop a PDF, Markdown or text file into `data/my/` and run the cell. UTF-8 is assumed for text; two-column PDFs extract
badly (compare the text before you trust the answers).

In [9]:
my = sorted(p for p in Path("data/my").glob("*") if p.suffix.lower() in (".pdf", ".md", ".txt")) if Path("data/my").exists() else []
if not my:
    print("Put a .pdf, .md or .txt file into data/my/ and run this cell again.")
else:
    my_chunks = []
    for p in my:
        my_chunks += chunk_text(load_document(str(p)), source=p.stem)
    my_by_id = {c.id: c for c in my_chunks}
    my_index = build_index([c.id for c in my_chunks], [c.text for c in my_chunks], model=EMBED_MODEL)
    print(len(my_chunks), "chunks. Ask a question:")
    q = input("Question: ") if os.environ.get("WORKSHOP_AUTO_APPROVE") != "1" else "What is this document about?"
    text, used = answer(q, my_index, my_by_id, k=K, model=MODEL)
    show_markdown(text)

Put a .pdf, .md or .txt file into data/my/ and run this cell again.


## 04.10 Exercises

🔧 **Run and change:** set `K = 1` and re-run 04.08. Which questions break? Then try `size=150` in 04.02.

🔧 **Run and change:** ask the same question in German. Does retrieval over English documents still find the passage?

🚀 **Stretch:** rerank. Ask the chat model, with `format=` and a tiny Pydantic model `{"relevant": bool, "reason": str}`, whether each
of the top 10 chunks answers the question; keep only the relevant ones and answer again. Compare with 04.05.

🚀 **Stretch:** index this workshop's website pages (download them as Markdown) and ask it about itself.

## ✅ Checkpoint

- Documents became chunks with sources; chunks became vectors; questions retrieve the chunks that matter
- Answers cite the passages and refuse when the corpus is silent
- You know what the context budget is and what happens when it is exceeded
- You judged five answers against expectations instead of trusting them

Next: `05_tool_use.ipynb`.